In [3]:
import pandas as pd
from mp_api.client import MPRester
API_KEY = "sDNN35D6Mt9njlSQFKlnmXr7dgxM9Ouw" # Add API_KEY from Materials Project Website -> It's different for every user and MP recommends its not shared
mpr = MPRester(API_KEY)

In [ ]:
docs = mpr.materials.insertion_electrodes.search(
        working_ion=["Li","Na","Mg","Ca","Ag"], # We can change this to find data for other types of batteries using working ions like Na, Ca ions etc.
        fields=[
            "battery_id",
            "formula_charge",
            "formula_discharge",
            "average_voltage",
            "capacity_grav",
            "capacity_vol",
            "energy_vol",
            "energy_grav",
            "stability_charge",
            "stability_discharge",
            "battery_type",
            "battery_formula",
            "elements",
            "host_structure",
            "max_delta_volume",
            "working_ion",
            "id_charge",
            "id_discharge",
            "nelements",
         ] # There are a lot more properties available, but not all of them seem to be useful and to save space, it's better to choose the necessary ones.
       )

data = [doc.dict() for doc in docs]
df = pd.DataFrame(data)




In [ ]:
df.to_csv("mp_battery_data.csv", index=False) # Save data to a csv file
#print(df.head())

In [ ]:
discharge_ids = df['id_discharge'].unique().tolist()

extra_docs = mpr.materials.summary.search(
    material_ids=discharge_ids,
    fields=[
        "material_id",
        "band_gap",
        "formation_energy_per_atom", # This is your energy of formation
        "density",                   # Physical density
        "structure"             # Cubic, Monoclinic, etc.
    ]
)

extra_df = pd.DataFrame([doc.dict() for doc in extra_docs])
df = df.merge(extra_df, left_on='id_discharge', right_on='material_id', how='left')

In [4]:
test_data = pd.read_csv("test_data_paper.csv") # downloading the large test file containing information from nearly 200,000 battery materials

In [5]:
test_data.head()

,No,id_charge,id_discharge,working_ion,average_voltage,max_delta_volume_per,capacity_grav,energy_grav
0,0,mp-1,mp-1016231,Mg,-0.175966,36.004713,781.306652,-137.483461
1,1,mp-1,mp-1183945,Rb,-0.019232,238.824165,206.531251,-3.972079
2,2,mp-1,mp-1183958,K,-0.032541,190.740857,321.360236,-10.457389
3,3,mp-1,mp-984762,Y,-0.465008,18.704805,362.490310,-168.560829
4,4,mp-1,mp-1184016,Rb,-0.043412,240.327129,206.531251,-8.965891


In [6]:
test_data["max_delta_volume"] = test_data["max_delta_volume_per"]/100 # converting percentage to actual value

In [ ]:
test_discharge_ids = test_data['id_discharge'].unique().tolist()
test_discharge_ids_sub = test_discharge_ids[:5000] # choosing the first 5000 batteries - we can change the number and even select random materials
extra_docs = mpr.materials.summary.search(
    material_ids=test_discharge_ids_sub,
    fields=[
        "material_id",
        "formula_pretty",
        "band_gap",
        "formation_energy_per_atom", # This is your energy of formation
        "density",                   # Physical density
        "structure",             # Cubic, Monoclinic, etc.
        "energy_above_hull",

    ]
)

extra_df = pd.DataFrame([doc.dict() for doc in extra_docs])
test_data_copy = test_data.iloc[:5000].copy()
test_data_copy = pd.concat([test_data_copy, extra_df], axis=1)

In [10]:
test_data_copy = test_data_copy.drop(columns=["max_delta_volume_per", "No", "id_charge", "id_discharge", "material_id","fields_not_requested"])

In [11]:
test_data_copy.head()

,working_ion,average_voltage,capacity_grav,energy_grav,max_delta_volume,formula_pretty,density,structure,formation_energy_per_atom,energy_above_hull,band_gap
0,Mg,-0.175966,781.306652,-137.483461,0.360047,CsMg3,2.203340,"{'@module': 'pymatgen.core.structure', '@class...",0.276177,0.276177,0.0
1,Rb,-0.019232,206.531251,-3.972079,2.388242,CsRb3,1.600529,"{'@module': 'pymatgen.core.structure', '@class...",0.027286,0.027286,0.0
2,K,-0.032541,321.360236,-10.457389,1.907409,CsK3,1.192071,"{'@module': 'pymatgen.core.structure', '@class...",0.018994,0.018994,0.0
3,Y,-0.465008,362.490310,-168.560829,0.187048,CsY,3.833530,"{'@module': 'pymatgen.core.structure', '@class...",0.716882,0.716882,0.0
4,Rb,-0.043412,206.531251,-8.965891,2.403271,CsRb3,1.590429,"{'@module': 'pymatgen.core.structure', '@class...",0.045421,0.045421,0.0


In [13]:
test_data_copy.to_csv("mp_battery_test_data.csv")